<a href="https://colab.research.google.com/github/pxtroniwnl/barcelona-de-indias-time-serie/blob/main/direccion_vientoIDEAM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dirección del viento — estaciones IDEAM cercanas a la laguna

Este notebook identifica las estaciones más cercanas a la laguna y construye sus series de dirección del viento conservando los valores observados. No se imputan datos, no se interpola, no se suaviza y no se agregan mediciones a otra frecuencia.

La base original se conserva en `df_raw`. Las únicas filas excluidas de la copia de trabajo son copias completamente idénticas, según la decisión tomada antes de construir el notebook.

## 1. Importación de librerías

Usaremos solamente herramientas comunes: `pandas` para tablas, `numpy` para cálculos y `matplotlib` para gráficas y mapa.

In [ ]:
from pathlib import Path
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except ImportError:
    display = print  # Permite ejecutar también como script fuera de Jupyter.

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

RUTA_CSV = Path("Dirección_del_Viento_20260818_SOLO_BOLIVAR.csv")
FORMATO_FECHA = "%Y %b %d %I:%M:%S %p"
PASO_ESPERADO_MIN = 10

print("Archivo encontrado:", RUTA_CSV.exists())

**Interpretación.** Si aparece `Archivo encontrado: True`, el notebook está ubicado junto al CSV y puede continuar.

## 2. Carga de la base original

Los códigos se cargan como texto para conservar sus ceros iniciales. `df_raw` no se sobrescribirá ni se limpiará: será nuestra copia fiel de la fuente.

In [ ]:
df_raw = pd.read_csv(
    RUTA_CSV,
    dtype={"CodigoEstacion": "string", "CodigoSensor": "string"},
)

FILAS_ORIGINALES = len(df_raw)
COLUMNAS_ORIGINALES = df_raw.columns.tolist()

print(f"Dimensiones: {df_raw.shape[0]:,} filas × {df_raw.shape[1]} columnas")
display(df_raw.head(3))

**Interpretación.** La base debe contener 2.864.494 registros y 12 columnas. Cada fila representa una observación de dirección del viento asociada a una estación y una fecha.

## 3. Exploración básica

Revisamos columnas, tipos, categorías principales, fechas y posibles celdas vacías antes de modificar cualquier dato. La conversión de fechas se guarda en una serie auxiliar y no cambia `df_raw`.

In [ ]:
descripcion_columnas = pd.DataFrame({
    "columna": df_raw.columns,
    "tipo_cargado": df_raw.dtypes.astype(str).values,
    "faltantes": df_raw.isna().sum().values,
})
display(descripcion_columnas)

fechas_aux = pd.to_datetime(
    df_raw["FechaObservacion"], format=FORMATO_FECHA, errors="coerce"
)
valores_aux = pd.to_numeric(df_raw["ValorObservado"], errors="coerce")

print("Periodo total:", fechas_aux.min(), "a", fechas_aux.max())
print("Fechas no convertibles:", fechas_aux.isna().sum())
print("Valores no convertibles:", valores_aux.isna().sum())
print("Rango observado:", valores_aux.min(), "a", valores_aux.max(), "grados")
print("Códigos de estación:", df_raw["CodigoEstacion"].nunique())

for columna in ["CodigoSensor", "DescripcionSensor", "UnidadMedida"]:
    print(f"{columna}:", sorted(df_raw[columna].dropna().astype(str).str.strip().unique()))

**Interpretación.** El periodo completo va del 15-may-2013 al 17-ago-2026. No hay celdas vacías ni errores de conversión y todos los valores están entre 0° y 360°. Las variantes de nombre y unidad son diferencias de escritura; no justifican alterar las mediciones.

## 4. Diagnóstico de duplicados y valores límite

Primero contamos filas idénticas y casos donde una estación tiene más de un valor en el mismo instante. También contamos 0° y 360°: ambos representan el norte, por lo que no deben tratarse automáticamente como valores atípicos.

In [ ]:
n_duplicados_exactos = int(df_raw.duplicated().sum())

revision_conflictos = pd.DataFrame({
    "CodigoEstacion": df_raw["CodigoEstacion"],
    "FechaObservacion": fechas_aux,
    "ValorObservado": valores_aux,
})
conflictos = (
    revision_conflictos.groupby(["CodigoEstacion", "FechaObservacion"])["ValorObservado"]
    .nunique()
    .loc[lambda s: s > 1]
)

print(f"Copias exactamente duplicadas: {n_duplicados_exactos:,}")
print(f"Instantes con valores contradictorios: {len(conflictos):,}")
print(f"Observaciones iguales a 0°: {(valores_aux == 0).sum():,}")
print(f"Observaciones iguales a 360°: {(valores_aux == 360).sum():,}")
display(conflictos.head(10).rename("cantidad_de_valores_distintos").reset_index())

**Interpretación.** Hay 164.960 copias idénticas y 19 instantes con valores distintos para el mismo código y fecha. Estos conflictos pertenecen a otras estaciones y se muestran para no ocultar el problema; no se resolverán automáticamente.

## 5. Catálogo de estaciones

Construimos una fila por código. El nombre mostrado es el más frecuente y la coordenada representativa es la más reciente. También contamos cuántas coordenadas distintas aparecen, porque algunos códigos cambiaron de ubicación durante su historia.

In [ ]:
base_catalogo = df_raw[[
    "CodigoEstacion", "NombreEstacion", "Municipio",
    "Latitud", "Longitud", "FechaObservacion"
]].copy()
base_catalogo["Fecha"] = fechas_aux

# Nombre más frecuente de cada código (solo para mostrarlo en tablas y gráficas).
nombres = (
    base_catalogo.groupby("CodigoEstacion")["NombreEstacion"]
    .agg(lambda s: s.astype(str).str.strip().value_counts().index[0])
    .rename("NombreEstacion")
)
municipios = (
    base_catalogo.groupby("CodigoEstacion")["Municipio"]
    .agg(lambda s: s.astype(str).str.strip().value_counts().index[0])
    .rename("Municipio")
)

resumen = base_catalogo.groupby("CodigoEstacion").agg(
    registros=("Fecha", "size"),
    inicio=("Fecha", "min"),
    fin=("Fecha", "max"),
    coordenadas_distintas=("Latitud", lambda s: len(set(zip(
        s.round(8), base_catalogo.loc[s.index, "Longitud"].round(8)
    )))),
)

ultima_coordenada = (
    base_catalogo.sort_values("Fecha")
    .drop_duplicates("CodigoEstacion", keep="last")
    .set_index("CodigoEstacion")[["Latitud", "Longitud"]]
)

catalogo = (
    pd.concat([nombres, municipios, ultima_coordenada, resumen], axis=1)
    .reset_index()
)

display(catalogo.sort_values("registros", ascending=False))

**Interpretación.** Se identifican 12 códigos. `EL GUAMO` cambia de coordenadas de forma importante en 2025; por eso las distancias usan la coordenada más reciente y la tabla deja visible el número de ubicaciones registradas. Los códigos seleccionados para la laguna mantienen coordenadas estables.

## 6. Región de interés y distancia Haversine

La región es el polígono de la laguna. Para representar el área mediante un punto calculamos su centroide. Haversine estima la distancia más corta sobre una Tierra esférica a partir de latitudes y longitudes.

In [ ]:
ROI_COORDS = [
    (-75.476052, 10.517524),
    (-75.476117, 10.518747),
    (-75.473158, 10.519223),
    (-75.470516, 10.525108),
    (-75.469572, 10.524876),
    (-75.471686, 10.518916),
    (-75.468394, 10.517219),
    (-75.468952, 10.516459),
]

def centroide_poligono(coordenadas):
    """Devuelve (latitud, longitud) del centroide de un polígono pequeño."""
    area_doble = 0.0
    suma_x = 0.0
    suma_y = 0.0
    pares = list(zip(coordenadas, coordenadas[1:] + coordenadas[:1]))
    for (x0, y0), (x1, y1) in pares:
        cruz = x0 * y1 - x1 * y0
        area_doble += cruz
        suma_x += (x0 + x1) * cruz
        suma_y += (y0 + y1) * cruz
    lon = suma_x / (3 * area_doble)
    lat = suma_y / (3 * area_doble)
    return lat, lon

def haversine_km(latitud, longitud, latitud_roi, longitud_roi):
    """Distancia sobre una esfera entre una estación y el ROI, en kilómetros."""
    radio_tierra = 6371.0088
    lat1 = np.radians(np.asarray(latitud, dtype=float))
    lon1 = np.radians(np.asarray(longitud, dtype=float))
    lat2 = math.radians(latitud_roi)
    lon2 = math.radians(longitud_roi)
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * math.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * radio_tierra * np.arcsin(np.sqrt(a))

LAT_ROI, LON_ROI = centroide_poligono(ROI_COORDS)
catalogo["distancia_km"] = haversine_km(
    catalogo["Latitud"], catalogo["Longitud"], LAT_ROI, LON_ROI
).round(2)
catalogo_distancias = catalogo.sort_values("distancia_km").reset_index(drop=True)

print(f"Centroide del ROI: {LAT_ROI:.6f}, {LON_ROI:.6f}")
display(catalogo_distancias[[
    "CodigoEstacion", "NombreEstacion", "Municipio",
    "Latitud", "Longitud", "distancia_km",
    "registros", "inicio", "fin"
]])

**Interpretación.** Los dos primeros códigos pertenecen al mismo sitio físico, Rafael Núñez, pero a periodos consecutivos. Para seleccionar dos sitios independientes usaremos el código vigente de Rafael Núñez (`0014015080`, 9,35 km) y UNAD (`1206500136`, 13,99 km). El código histórico del aeropuerto no se unirá.

## 7. Selección explícita y mapa

La selección se declara por código para que sea reproducible. El mapa es deliberadamente sencillo y local: muestra coordenadas, el polígono y las distancias sin depender de servicios externos.

In [ ]:
CODIGOS_SELECCIONADOS = ["0014015080", "1206500136"]
seleccion_estaciones = (
    catalogo_distancias[catalogo_distancias["CodigoEstacion"].isin(CODIGOS_SELECCIONADOS)]
    .set_index("CodigoEstacion")
    .loc[CODIGOS_SELECCIONADOS]
    .reset_index()
)
display(seleccion_estaciones[["CodigoEstacion", "NombreEstacion", "Latitud", "Longitud", "distancia_km"]])

fig, ax = plt.subplots(figsize=(8, 6))
lon_roi = [p[0] for p in ROI_COORDS] + [ROI_COORDS[0][0]]
lat_roi = [p[1] for p in ROI_COORDS] + [ROI_COORDS[0][1]]
ax.fill(lon_roi, lat_roi, color="#63c7da", alpha=0.45, label="Laguna (ROI)")
ax.scatter(LON_ROI, LAT_ROI, color="navy", marker="x", s=70, label="Centroide")

colores = ["#d1495b", "#2a9d8f"]
for (_, fila), color in zip(seleccion_estaciones.iterrows(), colores):
    ax.scatter(fila["Longitud"], fila["Latitud"], s=75, color=color)
    ax.plot([LON_ROI, fila["Longitud"]], [LAT_ROI, fila["Latitud"]],
            linestyle="--", linewidth=1, color=color)
    etiqueta = f'{fila["NombreEstacion"]}\n{fila["CodigoEstacion"]} — {fila["distancia_km"]:.2f} km'
    ax.annotate(etiqueta, (fila["Longitud"], fila["Latitud"]),
                xytext=(7, 4), textcoords="offset points", fontsize=8)

ax.set_xlabel("Longitud")
ax.set_ylabel("Latitud")
ax.set_title("Laguna y estaciones de dirección del viento seleccionadas")
ax.set_aspect(1 / math.cos(math.radians(LAT_ROI)))
ax.grid(alpha=0.25)
ax.legend(loc="best")
plt.tight_layout()
plt.show()

**Interpretación.** El mapa permite comprobar que ambas estaciones están al suroeste de la laguna. La cercanía geográfica apoya su selección, pero no garantiza que reproduzcan exactamente el microclima de la lámina de agua.

## 8. Copia de trabajo y limpieza mínima

Extraemos únicamente las estaciones elegidas y convertimos fecha y valor en la copia. Después conservamos una sola aparición de cada fila completamente idéntica. No se elimina ningún dato por su magnitud y `df_raw` permanece intacto.

In [ ]:
seleccion_raw = df_raw[
    df_raw["CodigoEstacion"].isin(CODIGOS_SELECCIONADOS)
].copy()

duplicados_por_estacion = (
    seleccion_raw.assign(es_duplicado=seleccion_raw.duplicated())
    .groupby("CodigoEstacion")["es_duplicado"]
    .sum()
    .astype(int)
    .rename("copias_excluidas")
    .reset_index()
)

serie_trabajo = seleccion_raw.drop_duplicates().copy()
serie_trabajo["FechaObservacion"] = pd.to_datetime(
    serie_trabajo["FechaObservacion"], format=FORMATO_FECHA, errors="coerce"
)
serie_trabajo["ValorObservado"] = pd.to_numeric(
    serie_trabajo["ValorObservado"], errors="coerce"
)
serie_trabajo = serie_trabajo.sort_values(
    ["CodigoEstacion", "FechaObservacion"]
).reset_index(drop=True)

conflictos_seleccion = (
    serie_trabajo.groupby(["CodigoEstacion", "FechaObservacion"])["ValorObservado"]
    .nunique()
    .loc[lambda s: s > 1]
)

display(duplicados_por_estacion)
print(f"Filas seleccionadas antes: {len(seleccion_raw):,}")
print(f"Filas después de retirar copias idénticas: {len(serie_trabajo):,}")
print("Conflictos en las estaciones seleccionadas:", len(conflictos_seleccion))
print("df_raw continúa intacto:", len(df_raw) == FILAS_ORIGINALES and df_raw.columns.tolist() == COLUMNAS_ORIGINALES)

**Interpretación.** Se excluyen 18.718 copias de Rafael Núñez y 21.104 de UNAD. No existen valores contradictorios para un mismo código y fecha en estas estaciones. La operación reduce conteos repetidos, pero no cambia ningún valor observado.

## 9. Cobertura temporal antes de graficar

Calculamos el paso entre registros consecutivos dentro de cada estación. La cobertura compara los tiempos observados con una rejilla teórica de 10 minutos; sirve para diagnosticar huecos, no para crear o rellenar mediciones.

In [ ]:
serie_trabajo["diferencia_min"] = (
    serie_trabajo.groupby("CodigoEstacion")["FechaObservacion"]
    .diff().dt.total_seconds().div(60)
)

cobertura = serie_trabajo.groupby("CodigoEstacion").agg(
    inicio=("FechaObservacion", "min"),
    fin=("FechaObservacion", "max"),
    observaciones=("FechaObservacion", "size"),
    paso_mediano_min=("diferencia_min", "median"),
    mayor_hueco_min=("diferencia_min", "max"),
).reset_index()
cobertura["tiempos_teoricos"] = (
    (cobertura["fin"] - cobertura["inicio"]).dt.total_seconds()
    .div(PASO_ESPERADO_MIN * 60).add(1).round().astype(int)
)
cobertura["cobertura_pct"] = (
    100 * cobertura["observaciones"] / cobertura["tiempos_teoricos"]
).round(2)
cobertura["mayor_hueco_dias"] = (cobertura["mayor_hueco_min"] / 1440).round(2)
display(cobertura.drop(columns="mayor_hueco_min"))

# Las rejillas quedan separadas de la serie; solo se usan para localizar tiempos ausentes.
rejillas_esperadas = {}
faltantes_por_estacion = {}
for codigo, grupo in serie_trabajo.groupby("CodigoEstacion"):
    rejilla = pd.date_range(grupo["FechaObservacion"].min(),
                            grupo["FechaObservacion"].max(), freq="10min")
    observados = pd.DatetimeIndex(grupo["FechaObservacion"].drop_duplicates())
    rejillas_esperadas[codigo] = rejilla
    faltantes_por_estacion[codigo] = rejilla.difference(observados)
    print(f"{codigo}: {len(faltantes_por_estacion[codigo]):,} tiempos ausentes; primeros 5:")
    print(faltantes_por_estacion[codigo][:5].tolist())

**Interpretación.** Rafael Núñez cubre aproximadamente 68,6 % de su ventana y UNAD cerca de 56,0 %. Los tiempos ausentes permanecen ausentes: la rejilla diagnóstica no se incorpora a `serie_trabajo`.

In [ ]:
# Diez interrupciones más largas por estación. Un salto de 10 minutos es normal.
huecos_largos = (
    serie_trabajo.loc[serie_trabajo["diferencia_min"] > PASO_ESPERADO_MIN,
                       ["CodigoEstacion", "FechaObservacion", "diferencia_min"]]
    .assign(
        fin_del_hueco=lambda d: d["FechaObservacion"],
        inicio_del_hueco=lambda d: d["FechaObservacion"] - pd.to_timedelta(d["diferencia_min"], unit="min"),
        tiempos_ausentes=lambda d: (d["diferencia_min"] / PASO_ESPERADO_MIN - 1).round().astype(int),
    )
    .sort_values(["CodigoEstacion", "diferencia_min"], ascending=[True, False])
    .groupby("CodigoEstacion")
    .head(10)
)
display(huecos_largos[["CodigoEstacion", "inicio_del_hueco", "fin_del_hueco",
                       "diferencia_min", "tiempos_ausentes"]])

**Interpretación.** UNAD presenta la interrupción más crítica: no tiene registros durante todo 2023. La tabla permite citar las mayores discontinuidades sin asumir que entre ellas hubo valores iguales a cero.

In [ ]:
# Conteo anual de observaciones; no se promedian ni modifican los valores.
cobertura_anual = (
    serie_trabajo.assign(anio=serie_trabajo["FechaObservacion"].dt.year)
    .pivot_table(index="anio", columns="CodigoEstacion",
                 values="ValorObservado", aggfunc="size", fill_value=0)
)
display(cobertura_anual)

**Interpretación.** La tabla anual hace visibles los periodos con poca información. Un cero en esta tabla significa que no hubo observaciones ese año; no es una dirección del viento imputada.

## 10. Serie temporal original

Graficamos cada observación como un punto diminuto. Esta representación evita inventar líneas continuas a través de los huecos y no reduce los datos a promedios horarios, diarios o mensuales.

In [ ]:
nombres_plot = seleccion_estaciones.set_index("CodigoEstacion")["NombreEstacion"].to_dict()
fig, axes = plt.subplots(2, 1, figsize=(15, 7), sharex=True, sharey=True)

for ax, codigo, color in zip(axes, CODIGOS_SELECCIONADOS, colores):
    sub = serie_trabajo[serie_trabajo["CodigoEstacion"] == codigo]
    ax.plot(sub["FechaObservacion"], sub["ValorObservado"],
            linestyle="None", marker=",", color=color, alpha=0.65)
    ax.set_title(f"{nombres_plot[codigo]} — {codigo}", loc="left", fontsize=10)
    ax.set_ylabel("Dirección (°)")
    ax.set_ylim(-5, 365)
    ax.set_yticks([0, 90, 180, 270, 360])
    ax.grid(alpha=0.2)

axes[-1].set_xlabel("Fecha de observación")
fig.suptitle("Dirección del viento — observaciones originales de las estaciones seleccionadas")
plt.tight_layout()
plt.show()

**Interpretación.** Los saltos visuales entre valores próximos a 0° y 360° no representan necesariamente cambios bruscos: ambos extremos corresponden al norte. Los espacios sin puntos son periodos sin observaciones y se conservan como tales.

## 11. Estadísticas básicas y carácter circular

La dirección no es una variable lineal: el promedio ordinario entre 359° y 1° sería 180°, aunque ambas observaciones apuntan al norte. Por eso mostramos los estadísticos lineales solo como descripción del archivo y calculamos aparte la media circular. Ningún resultado reemplaza los datos originales.

In [ ]:
filas_estadisticas = []
for codigo, grupo in serie_trabajo.groupby("CodigoEstacion"):
    valores = grupo["ValorObservado"].dropna()
    angulos = np.radians(valores)
    promedio_seno = np.sin(angulos).mean()
    promedio_coseno = np.cos(angulos).mean()
    media_circular = np.degrees(np.arctan2(promedio_seno, promedio_coseno)) % 360
    concentracion = np.sqrt(promedio_seno ** 2 + promedio_coseno ** 2)
    filas_estadisticas.append({
        "CodigoEstacion": codigo,
        "n": len(valores),
        "minimo": valores.min(),
        "q25_lineal": valores.quantile(0.25),
        "mediana_lineal": valores.median(),
        "q75_lineal": valores.quantile(0.75),
        "maximo": valores.max(),
        "media_circular_grados": media_circular,
        "concentracion_circular_0_a_1": concentracion,
    })

estadisticas = pd.DataFrame(filas_estadisticas).round(2)
display(estadisticas)

**Interpretación.** La media circular resume el rumbo dominante respetando que 0° y 360° son equivalentes. La concentración se acerca a 1 cuando los rumbos son muy parecidos y a 0 cuando están muy dispersos; no debe confundirse con velocidad del viento.

## 12. Conclusiones y limitaciones

- Las estaciones elegidas son las dos ubicaciones físicas diferentes más cercanas al centroide de la laguna.
- Rafael Núñez es una estación aeroportuaria y UNAD es urbana; cercanía no garantiza representatividad exacta del microclima lagunar.
- Ambas estaciones presentan huecos importantes y UNAD carece de datos durante todo 2023.
- El código histórico `0014015020` parece corresponder al mismo sitio de Rafael Núñez, pero no fue concatenado con `0014015080`.
- Las fechas se mantienen sin zona horaria porque el archivo no declara una.
- La serie conserva los grados observados: no se imputaron, interpolaron, suavizaron, agregaron ni eliminaron valores atípicos.
- Cualquier agregación temporal, unión de códigos o tratamiento adicional deberá justificarse y decidirse antes de aplicarse.